In [15]:
# Step 1: Imports and data loading
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 55

df_select = pd.read_parquet(
    "train_preprocessed.parquet",
    engine="fastparquet",
)

df_select.shape

(1647591, 47)

In [16]:
df_select.head()

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.345572,20296.00,16.83,63.36,10424.464262,10424.464262,0.0,0.0,0.0,0.0,...,0.248498,50.10,8.309762e+07,9.5,10.006691,0.352186,0.624737,0.10,141.55,DDoS-UDP_Flood
1,4.101233,118.28,6.00,63.58,0.530194,0.530194,0.0,0.0,0.0,0.0,...,4.233164,55.52,8.294707e+07,9.5,10.499854,5.998103,165.132926,0.11,141.55,DoS-TCP_Flood
2,0.031695,54.92,6.11,64.00,0.755846,0.755846,0.0,0.0,0.0,0.0,...,0.529146,54.19,8.307590e+07,9.5,10.406318,0.749763,2.580202,0.11,141.55,DDoS-TCP_Flood
3,0.189679,156.06,6.00,64.00,15.017715,15.017715,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.336203e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-SynonymousIP_Flood
4,0.007134,59.12,6.00,64.00,7.550638,7.550638,0.0,0.0,0.0,0.0,...,6.448502,56.32,8.294290e+07,9.5,10.575549,9.135028,347.875039,0.13,141.55,DoS-TCP_Flood


In [17]:
#after feature selection,top features are 
df_process=df_select[['IAT', 'Min', 'Magnitue', 'fin_flag_number', 'psh_flag_number', 'syn_flag_number', 'Tot sum', 'Protocol Type', 'ICMP', 'Header_Length', 'rst_count', 'Radius', 'fin_count', 'syn_count', 'flow_duration', 'Srate', 'Number', 'AVG', 'Rate', 'Variance', 'HTTPS', 'urg_count', 'Duration', 'Weight', 'HTTP', 'Max', 'Tot size', 'Covariance', 'ack_count', 'Std', 'rst_flag_number', 'UDP', 'ack_flag_number', 'SSH', 'TCP', 'LLC','label']]
df_process.head()

,IAT,Min,Magnitue,fin_flag_number,psh_flag_number,syn_flag_number,Tot sum,Protocol Type,ICMP,Header_Length,...,Covariance,ack_count,Std,rst_flag_number,UDP,ack_flag_number,SSH,TCP,LLC,label
0,8.309762e+07,50.0,10.006691,0.0,0.0,0.0,526.00,16.83,0.0,20296.00,...,0.624737,0.00,0.248498,0.0,1.0,0.0,0.0,0.0,1.0,DDoS-UDP_Flood
1,8.294707e+07,54.0,10.499854,0.0,0.0,0.0,583.72,6.00,0.0,118.28,...,165.132926,0.00,4.233164,0.0,0.0,0.0,0.0,1.0,1.0,DoS-TCP_Flood
2,8.307590e+07,54.0,10.406318,0.0,0.0,0.0,569.09,6.11,0.0,54.92,...,2.580202,0.00,0.529146,0.0,0.0,0.0,0.0,1.0,1.0,DDoS-TCP_Flood
3,8.336203e+07,54.0,10.392305,0.0,0.0,1.0,567.00,6.00,0.0,156.06,...,0.000000,0.00,0.000000,0.0,0.0,0.0,0.0,1.0,1.0,DDoS-SynonymousIP_Flood
4,8.294290e+07,54.0,10.575549,0.0,0.0,0.0,594.84,6.00,0.0,59.12,...,347.875039,0.01,6.448502,0.0,0.0,0.0,0.0,1.0,1.0,DoS-TCP_Flood


In [21]:
# Step 3: Prepare X (36 features) and y (label)
FEATURE_COLUMNS = [
    'IAT', 'Min', 'Magnitue', 'fin_flag_number', 'psh_flag_number', 'syn_flag_number',
    'Tot sum', 'Protocol Type', 'ICMP', 'Header_Length', 'rst_count', 'Radius',
    'fin_count', 'syn_count', 'flow_duration', 'Srate', 'Number', 'AVG', 'Rate',
    'Variance', 'HTTPS', 'urg_count', 'Duration', 'Weight', 'HTTP', 'Max', 'Tot size',
    'Covariance', 'ack_count', 'Std', 'rst_flag_number', 'UDP', 'ack_flag_number',
    'SSH', 'TCP', 'LLC'
]

# X uses exactly the required 36 feature columns (in fixed order)
# float32 reduces memory usage significantly vs float64
X = df_process[FEATURE_COLUMNS].astype(np.float32).copy()

# y uses the dataset label column as requested
y_full_label = df_process['label'].astype(str).str.strip()

# Dataset-aware binary mapping:
# 0 = Normal (BenignTraffic), 1 = Attack (all other classes including DDoS/DoS/Mirai/etc.)
def map_label_to_binary(lbl: str) -> int:
    normalized = lbl.lower()
    return 0 if normalized in {'benigntraffic', 'benign', 'normal'} else 1

y = y_full_label.apply(map_label_to_binary).astype(np.int8)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y,
)

# Class imbalance helper for XGBoost
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = max(1.0, neg_count / max(pos_count, 1))

print('Label counts (original):')
print(y_full_label.value_counts().head(10))
print('\nBinary counts (0=Normal, 1=Attack):')
print(y.value_counts())
print('\nX_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('Attack ratio in y_train:', y_train.mean())
print('Attack ratio in y_val:', y_val.mean())
print('scale_pos_weight:', scale_pos_weight)

Label counts (original):
label
DDoS-ICMP_Flood            254426
DDoS-UDP_Flood             191267
DDoS-TCP_Flood             158550
DDoS-PSHACK_Flood          144376
DDoS-SYN_Flood             143596
DDoS-RSTFINFlood           142632
DDoS-SynonymousIP_Flood    126625
DoS-UDP_Flood              117127
DoS-TCP_Flood               94252
DoS-SYN_Flood               71272
Name: count, dtype: int64

Binary counts (0=Normal, 1=Attack):
label
1    1608730
0      38861
Name: count, dtype: int64

X_train shape: (1318072, 36)
X_val shape: (329519, 36)
Attack ratio in y_train: 0.9764132763612307
Attack ratio in y_val: 0.9764141066220764
scale_pos_weight: 1.0


In [ ]:
# Step 4: Memory-safe hyperparameter search for XGBoost
# Tune on a stratified subset to avoid RAM paging errors on very large datasets.
X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=200000,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

# Recompute inside this cell so it works even if cells are run out-of-order.
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = max(1.0, neg_count / max(pos_count, 1))

base_xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=2,
    scale_pos_weight=scale_pos_weight,
)

param_grid = {
    'n_estimators': [120, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'gamma': [0.0, 0.1, 0.3],
    'reg_lambda': [1, 3, 5],
}

random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_grid,
    n_iter=10,
    scoring='accuracy',
    cv=3,
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

random_search.fit(X_tune, y_tune)

print('Tuning subset shape:', X_tune.shape)
print('scale_pos_weight used:', scale_pos_weight)
print('Best CV accuracy:', random_search.best_score_)
print('Best hyperparameters:', random_search.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END colsample_bytree=1.0, gamma=0.0, learning_rate=0.03, max_depth=8, min_child_weight=1, n_estimators=300, reg_lambda=3, subsample=0.8; total time=   4.5s
[CV] END colsample_bytree=1.0, gamma=0.0, learning_rate=0.03, max_depth=8, min_child_weight=1, n_estimators=300, reg_lambda=3, subsample=0.8; total time=   4.3s
[CV] END colsample_bytree=1.0, gamma=0.0, learning_rate=0.03, max_depth=8, min_child_weight=1, n_estimators=300, reg_lambda=3, subsample=0.8; total time=   4.4s


In [ ]:
# Step 5: Train final model with optimal parameters and show model accuracy
best_params = random_search.best_params_

# Recompute inside this cell so it works even if Step 4 was skipped.
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = max(1.0, neg_count / max(pos_count, 1))

final_model = XGBClassifier(
    **best_params,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=2,
)

final_model.fit(X_train, y_train)

# Predict on validation first (faster, memory-safe)
val_pred = final_model.predict(X_val)
val_acc = accuracy_score(y_val, val_pred)

print('--- Evaluating Model ---')
print(f'Accuracy: {val_acc:.4f}')
print('\nClassification Report:\n')
print(classification_report(y_val, val_pred, target_names=['Benign (0)', 'Attack (1)'], digits=4))

print('\nConfusion Matrix:\n', confusion_matrix(y_val, val_pred))
print('Best CV Accuracy from search:', f'{random_search.best_score_:.6f}', f'({random_search.best_score_*100:.2f}%)')

In [ ]:
# Step 6: Backend-safe wrapper with strict feature order and output format
class BackendSafeDDoSModel:
    def __init__(self, model, feature_columns):
        self.model = model
        self.feature_columns = list(feature_columns)

    def predict(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError('Input must be a pandas DataFrame.')

        missing_cols = [c for c in self.feature_columns if c not in X.columns]
        if missing_cols:
            raise ValueError(f'Missing required columns: {missing_cols}')

        X_ordered = X[self.feature_columns]
        preds = self.model.predict(X_ordered)

        # Must be numpy array of integers with shape (n,)
        return np.asarray(preds, dtype=np.int64)

In [ ]:
# Step 7: Save model as .pkl for API layer
export_model = BackendSafeDDoSModel(final_model, FEATURE_COLUMNS)
joblib.dump(export_model, 'model.pkl')

print('--- Saving Model ---')
print('Model saved to: model.pkl')
print('Expected class mapping -> 1: Attack, 0: Benign')

In [ ]:
# Step 8: API layer inference helper

def predict_for_api(sample_df, model_path='model.pkl'):
    model_obj = joblib.load(model_path)
    pred = model_obj.predict(sample_df)  # numpy array, shape (1,)
    response = {
        'prediction': int(pred[0]),
        'result': 'Attack' if int(pred[0]) == 1 else 'Benign'
    }
    return pred, response

# Example usage:
# sample = X_val.iloc[[0]].copy()
# pred_array, api_response = predict_for_api(sample)
# print(pred_array)      # e.g., [1]
# print(api_response)    # {'prediction': 1, 'result': 'Attack'}